
# 02. Reducción dimensional



## Objetivo

Estudiar las cinco estrategias utilizadas para reducir las características
candidatas:

1. Importancia mediante Random Forest.
2. RFE.
3. Selección de lags.
4. PCA.
5. Método combinado.

La reducción dimensional genera diferentes datasets que posteriormente
serán utilizados para entrenar las MLP.


In [ ]:

from pathlib import Path
from datetime import datetime

RUTA_PROYECTO = Path(
    r"C:\Users\marco\Documentos\investigacion"
    r"\machine_learning_idalina\6_redes_neuronales"
)

RUTA_DATOS_RAW = RUTA_PROYECTO / "2_datos" / "1_raw"

RUTA_PROCESADOS = RUTA_PROYECTO / "2_datos" / "2_procesados"

RUTA_RESULTADOS = RUTA_PROCESADOS / "resultados_mlp"

RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Proyecto:")
print(RUTA_PROYECTO)

print("\nDatos originales:")
print(RUTA_DATOS_RAW)

print("\nDatos procesados:")
print(RUTA_PROCESADOS)

print("\nResultados MLP:")
print(RUTA_RESULTADOS)


## 1. Bibliotecas

In [ ]:

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA


## 2. Cargar datos

In [ ]:

RUTA_EXCEL = (
    RUTA_DATOS_RAW /
    "2_meteo_epi_2021-2026_1_rezagos.xlsx"
)

df = pd.read_excel(
    RUTA_EXCEL,
    parse_dates=["fecha"]
)

df = df.sort_values("fecha").reset_index(drop=True)

df = crear_features_avanzadas(df)
df = df.ffill().fillna(0)

COLUMNAS_NO_FEATURE = [
    "fecha",
    "casos_dengue"
]

features = [
    c for c in df.columns
    if c not in COLUMNAS_NO_FEATURE
]

print("Features candidatas:", len(features))


## 3. Selección mediante importancia de Random Forest

In [ ]:

def seleccionar_por_importancia(
    df,
    features,
    threshold=0.005
):

    X = df[features].values
    y = df["casos_dengue"].values

    imputer = SimpleImputer(strategy="median")

    X = imputer.fit_transform(X)

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )

    rf.fit(X, y)

    importancias = pd.DataFrame({
        "feature": features,
        "importance": rf.feature_importances_
    })

    importancias = (
        importancias
        .sort_values(
            "importance",
            ascending=False
        )
    )

    seleccionadas = (
        importancias[
            importancias["importance"] > threshold
        ]["feature"]
        .tolist()
    )

    return seleccionadas, importancias


## 4. RFE

In [ ]:

def seleccionar_rfe(
    df,
    features,
    n_features=20
):

    X = df[features].values
    y = df["casos_dengue"].values

    imputer = SimpleImputer(
        strategy="median"
    )

    X = imputer.fit_transform(X)

    rf = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=42
    )

    selector = RFE(
        rf,
        n_features_to_select=n_features,
        step=1
    )

    selector.fit(X, y)

    seleccionadas = [
        features[i]
        for i in range(len(features))
        if selector.support_[i]
    ]

    return seleccionadas


## 5. Lags óptimos

In [ ]:

def seleccionar_lags_optimos(df):

    lags = [
        f"casos_dengue_lag_{i}"
        for i in range(1, 13)
        if f"casos_dengue_lag_{i}" in df.columns
    ]

    correlaciones = []

    X = df[lags].values
    y = df["casos_dengue"].values

    X = SimpleImputer(
        strategy="median"
    ).fit_transform(X)

    for i, col in enumerate(lags):

        corr = np.corrcoef(
            X[:, i],
            y
        )[0, 1]

        if not np.isnan(corr):

            correlaciones.append(
                (col, abs(corr))
            )

    correlaciones.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return [
        col
        for col, corr in correlaciones
        if corr > 0.3
    ]


## 6. PCA

In [ ]:

def aplicar_pca(
    df,
    features,
    n_components=10
):

    X = df[features].values

    X = SimpleImputer(
        strategy="median"
    ).fit_transform(X)

    scaler = RobustScaler()

    X_scaled = scaler.fit_transform(X)

    pca = PCA(
        n_components=n_components
    )

    X_pca = pca.fit_transform(X_scaled)

    columnas = [
        f"PC_{i+1}"
        for i in range(n_components)
    ]

    df_pca = pd.DataFrame(
        X_pca,
        columns=columnas
    )

    df_pca["fecha"] = df["fecha"].values
    df_pca["casos_dengue"] = (
        df["casos_dengue"].values
    )

    return (
        df_pca,
        columnas,
        pca
    )


## 7. Comparar las estrategias

In [ ]:

features_imp, importancias = (
    seleccionar_por_importancia(
        df,
        features
    )
)

features_rfe = seleccionar_rfe(
    df,
    features,
    n_features=20
)

features_lags = seleccionar_lags_optimos(df)

df_pca, features_pca, pca = aplicar_pca(
    df,
    features,
    n_components=10
)

comparacion = pd.DataFrame({
    "Método": [
        "Importancia",
        "RFE",
        "Lags",
        "PCA"
    ],

    "Features": [
        len(features_imp),
        len(features_rfe),
        len(features_lags),
        len(features_pca)
    ]
})

comparacion



### Observación metodológica

PCA no selecciona variables originales. Produce componentes principales.

Por tanto:

- Importancia → selección de features.
- RFE → selección de features.
- Lags → selección de features.
- PCA → transformación/reducción de dimensionalidad.
